In [2]:
# ==============================================================================
# CELL 1: DJANGO BOOTSTRAP — chạy cell này TRƯỚC TIÊN, mọi cell khác phụ thuộc
# ==============================================================================
import sys
import os

os.chdir("/home/duoc/app")  # chuyển về thư mục project
sys.path.insert(0, "/home/duoc/app")  # đảm bảo Python tìm thấy config/

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")
# Khai báo settings module trước khi import bất cứ thứ gì của Django
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")

import django

django.setup()  # Bắt buộc — khởi tạo App Registry trước khi dùng ORM

print("Django setup OK —", django.get_version())
print("Settings module  :", os.environ["DJANGO_SETTINGS_MODULE"])

import nest_asyncio

nest_asyncio.apply()


from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import RetryPolicy

from agent_os.system.infra.telemetry import setup_observability
from agent_os.system.shields.shield_faults import PipelineError
from agent_os.system.infra.persistence import factory
from app.core.main_bus import MainBus
from app.core.registry  import BusRegistry
from agent_os.system.shields.shield_node import Node

# =========================================================
# MAPPING: CORE AGENT FLOW (Theo sơ đồ Mermaid)
# =========================================================

from app.nodes_library.node_aggregator.adapter_aggregator import node_aggregator
from app.nodes_library.node_evaluator.adapter_evaluator import node_evaluator
from app.nodes_library.node_final_response.adapter_final_response import (
    node_final_response,
)
from app.nodes_library.node_human_review.adapter_human_review import (
    node_human_review,
)
from app.nodes_library.node_input_guard.adapter_input_guard import node_input_guard
from app.nodes_library.node_marketing.adapter_node_marketing import node_marketing
from app.nodes_library.node_output_guard.adapter_output_guard import (
    node_output_guard,
)
from app.nodes_library.node_shared_state.adapter_shared_state import (
    node_shared_state,
)
from app.nodes_library.node_supervisor.adapter_supervisor import node_supervisor
from app.nodes_library.node_cache_layer.adapter_cache_read import node_cache_read
from app.nodes_library.node_cache_layer.adapter_cache_write import node_cache_write
from app.nodes_library.node_heuristic_router.adapter_heuristic_router import (
    node_heuristic_router,
)
from app.nodes_library.node_knowledgebase.adapter_knowledgebase import (
    node_knowledgebase,
)
from app.nodes_library.node_generation.adapter_generation import node_generation
from app.nodes_library.node_relevance_check.adapter_relevance_check import (
    node_relevance_check,
)
from app.nodes_library.node_fallback_search.adapter_fallback_search import (
    node_fallback_search,
)

Django setup OK — 6.0.5
Settings module  : config.settings.local


In [ ]:
import sys
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# 1. BẢO VỆ JUPYTER: Xóa định nghĩa cũ khỏi RAM để tránh hoàn toàn lỗi trùng Node
if "optimized_board" in globals():
    del optimized_board

# Khai báo đồ thị mới tinh trên vùng nhớ sạch
optimized_board = StateGraph(MainBus)

# =============================================================================
# 2. ĐĂNG KÝ CÁC NODE (Khớp chuẩn xác tên hàm và định danh hệ thống của cậu)
# =============================================================================
Node(BusRegistry.IG, node_input_guard).mount(
    optimized_board
)  # Node 1: Kiểm tra an toàn
Node(BusRegistry.RO, node_heuristic_router).mount(
    optimized_board
)  # Node 2: Bộ định tuyến từ khóa
Node(BusRegistry.CA, node_cache_layer).mount(
    optimized_board
)  # Node 3: Bộ nhớ đệm L1/L2
Node(BusRegistry.CA, node_cache_layer).mount(
    optimized_board
)  # Node 3: Bộ nhớ đệm L1/L2
Node(BusRegistry.FR, node_final_response).mount(
    optimized_board
)  # Node 8: Trả kết quả UI
Node(BusRegistry.FB, node_fallback_search).mount(
    optimized_board
)  # Node THÊM MỚI: Khối cứu cánh Fallback

Node(BusRegistry.KLB, node_knowledgebase).mount(
    optimized_board
)  # Node 4: Quét FAISS lấy Context
Node(BusRegistry.RC, node_relevance_check).mount(
    optimized_board
)  # Node 5: Check rác tài liệu
Node(BusRegistry.LLM, node_generation).mount(
    optimized_board
)  # Node 6: Gọi LLM sinh câu trả lời
Node(BusRegistry.OG, node_output_guard).mount(
    optimized_board
)  # Node 7: Kiểm tra đầu ra

# =============================================================================
# 3. THIẾT LẬP ĐƯỜNG ĐI TUYẾN TÍNH - KHỚP 100% MÃ MERMAID CỦA CẬU
# =============================================================================

# Trục dọc ban đầu (Nét liền)
optimized_board.add_edge(START, BusRegistry.IG)  # __start__ --> input_guard
optimized_board.add_edge(
    BusRegistry.IG, BusRegistry.RO
)  # input_guard --> heuristic_router
optimized_board.add_edge(
    BusRegistry.RO, BusRegistry.CA
)  # heuristic_router --> cache_layer

# Rẽ nhánh điều kiện tại tầng Cache Layer (Nét đứt)
optimized_board.add_conditional_edges(
    BusRegistry.CA,
    lambda state: state.route("cache_status"),
    {
        "hit": BusRegistry.FR,  # cache_layer -. hit .-> final_response
        "miss": BusRegistry.KLB,  # cache_layer -. miss .-> knowledge_base
    },
)

# Trục dọc sau khi trượt cache (Nét liền)
optimized_board.add_edge(
    BusRegistry.KLB, BusRegistry.RC
)  # knowledge_base --> relevance_check

# Rẽ nhánh điều kiện tại tầng Relevance Check (Nét đứt)
optimized_board.add_conditional_edges(
    BusRegistry.RC,
    lambda state: state.route("relevance_status"),
    {
        "high_rel": BusRegistry.LLM,  # relevance_check -. high_rel .-> llm_generation
        "low_rel": BusRegistry.FB,  # relevance_check -. low_rel .-> fallback_search
    },
)

# Trục dọc sau khi RAG sạch (Nét liền)
optimized_board.add_edge(
    BusRegistry.LLM, BusRegistry.OG
)  # llm_generation --> output_guard
optimized_board.add_edge(
    BusRegistry.OG, BusRegistry.FR
)  # output_guard --> final_response

optimized_board.add_conditional_edges(
    BusRegistry.FB,  # Từ node Fallback
    lambda state: state.route("fallback_search"),  # Kiểm tra route
    {
        "tool_executed": BusRegistry.LLM,  # Đẩy kết quả qua LLM để viết câu trả lời
        "error": BusRegistry.FR,  # Hoặc đẩy qua Human Review nếu lỗi
    },
)

optimized_board.add_edge(BusRegistry.LLM, BusRegistry.FR)
optimized_board.add_edge(BusRegistry.FR, END)  # final_response --> __end__

# =============================================================================
# 4. COMPILE & DEBUG
# =============================================================================
main_v9 = optimized_board.compile(
    checkpointer=MemorySaver(), interrupt_before=[BusRegistry.FR]
)

if __name__ == "__main__":
    print("\n===== MAINBOARD STRUCTURE =====")
    graph = main_v9.get_graph()
    for node in graph.nodes:
        print(f"Node: {node}")
    print("\n✅ MAINBOARD READY - SƠ ĐỒ ĐÃ KHỚP CÔNG TY 100%")

In [ ]:
from IPython.display import display, Markdown

mermaid_code = main_v9.get_graph().draw_mermaid()

display(
    Markdown(f"""
```mermaid
{mermaid_code}

""")
)

In [ ]:
code = main_v9.get_graph().draw_mermaid()
print(code)